# Transfer Learning with CNNs: Flower Classification
3D Club Induction Task 2026 - Task 2

**Framework:** TensorFlow / Keras  |  **Pretrained backbone:** MobileNetV2 (ImageNet)  |  **Dataset:** TensorFlow Flowers (5 classes)

## 1. Problem Statement
Training a deep CNN from scratch needs lots of data, compute and time. Instead, we reuse a CNN pretrained on ImageNet as a feature extractor and adapt it to a new problem: classifying photos of flowers into 5 classes.

Plan:
1. Prepare data (stratified train / validation / test split, resizing, augmentation).
2. Replace the ImageNet classification head with a new one for 5 classes.
3. Train with the backbone **frozen** (feature extraction).
4. **Fine-tune** the top layers of the backbone at a low learning rate.
5. Evaluate both models (plus a small CNN trained from scratch as a reference) with accuracy, precision, recall, F1 and confusion matrices, and discuss the results.

## 2. Dataset and Architecture
- **Dataset:** the "flower_photos" collection released by the TensorFlow team: about 3,670 photos in 5 classes (daisy, dandelion, roses, sunflowers, tulips), downloaded automatically below.
- **Backbone:** MobileNetV2 (Sandler et al., 2018). It uses depthwise-separable convolutions and inverted residual blocks, giving a small, fast network (~2.3M parameters in the convolutional base) pretrained on ImageNet-1k.
- **New head:** Global Average Pooling -> Dropout(0.3) -> Dense(5, softmax).

In [ ]:
import os, pathlib, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 3. Dataset Preparation

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive = keras.utils.get_file("flower_photos", origin=url, untar=True)
data_dir = pathlib.Path(archive)
if not (data_dir / "daisy").exists():          # some versions return the parent folder
    data_dir = data_dir / "flower_photos"

class_names = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
NUM_CLASSES = len(class_names)
paths, labels = [], []
for idx, cname in enumerate(class_names):
    for f in (data_dir / cname).glob("*.jpg"):
        paths.append(str(f)); labels.append(idx)
paths, labels = np.array(paths), np.array(labels)
print(class_names, "| images:", len(paths))
print(pd.Series(labels).map(dict(enumerate(class_names))).value_counts())

In [ ]:
# Stratified 70 / 15 / 15 split so every class appears in each subset in the same proportion
p_train, p_tmp, y_train, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(len(p_train), len(p_val), len(p_test))

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)          # values stay in 0-255; scaling is done inside the model
    return img, label

def make_ds(p, y, training):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if training:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(p_train, y_train, True)
val_ds   = make_ds(p_val,   y_val,   False)
test_ds  = make_ds(p_test,  y_test,  False)   # not shuffled, so predictions line up with y_test

In [ ]:
plt.figure(figsize=(11, 6))
for imgs, labs in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(imgs[i].numpy().astype("uint8")); plt.title(class_names[labs[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

## 4. Transfer Learning: Building the Model
Data augmentation (random flip / rotation / zoom) is placed inside the model so it is active only during training. `mobilenet_v2.preprocess_input` rescales pixels to [-1, 1], which is what the pretrained weights expect.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

base_model = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False                       # freeze for phase 1

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)                  # keep BatchNorm in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
tl_model = keras.Model(inputs, outputs, name="mobilenetv2_transfer")
tl_model.summary()

## 5. Model Training
### 5a. Phase 1: frozen backbone (only the new head is trained)

In [ ]:
tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
hist_frozen = tl_model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb)

In [ ]:
def evaluate(model, name):
    probs = model.predict(test_ds, verbose=0)
    pred = probs.argmax(axis=1)
    acc = accuracy_score(y_test, pred)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    return {"model": name, "accuracy": acc, "macro precision": p, "macro recall": r, "macro F1": f}, pred

res_frozen, pred_frozen = evaluate(tl_model, "MobileNetV2 (frozen)")
print(res_frozen)

### 5b. Phase 2: fine-tuning
We unfreeze the last 30 layers of the backbone (higher-level, more task-specific features) and train with a **much lower learning rate (1e-5)** so the pretrained weights are adjusted gently rather than destroyed. BatchNorm layers stay frozen, which is the standard practice for stable fine-tuning with small batches.

In [ ]:
base_model.trainable = True
FINE_TUNE_FROM = len(base_model.layers) - 30
for i, layer in enumerate(base_model.layers):
    layer.trainable = (i >= FINE_TUNE_FROM) and not isinstance(layer, layers.BatchNormalization)
print("Trainable backbone layers:", sum(l.trainable for l in base_model.layers), "of", len(base_model.layers))

tl_model.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_ft = tl_model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb)

In [ ]:
res_ft, pred_ft = evaluate(tl_model, "MobileNetV2 (fine-tuned)")
print(res_ft)

### 5c. Reference: small CNN trained from scratch
To judge how much transfer learning helps, we train a compact CNN with random initialisation on the same data and split.

In [ ]:
scratch = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    augment,
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="small_cnn_scratch")
scratch.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_scratch = scratch.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb)
res_scratch, pred_scratch = evaluate(scratch, "Small CNN (from scratch)")
print(res_scratch)

In [ ]:
def curves(histories, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        ax[0].plot(h.history["accuracy"], label=f"{name} train"); ax[0].plot(h.history["val_accuracy"], "--", label=f"{name} val")
        ax[1].plot(h.history["loss"], label=f"{name} train"); ax[1].plot(h.history["val_loss"], "--", label=f"{name} val")
    ax[0].set_title("Accuracy"); ax[1].set_title("Loss")
    for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

curves({"frozen": hist_frozen, "fine-tune": hist_ft, "scratch": hist_scratch}, "Training curves (note: fine-tune epochs restart at 0)")

## 6. Model Evaluation

In [ ]:
results = pd.DataFrame([res_frozen, res_ft, res_scratch]).set_index("model")
results.round(4)

In [ ]:
for name, pred in [("MobileNetV2 (frozen)", pred_frozen), ("MobileNetV2 (fine-tuned)", pred_ft)]:
    print("=" * 60, "\n", name)
    print(classification_report(y_test, pred, target_names=class_names, digits=3))

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, (name, pred) in zip(ax, [("Frozen", pred_frozen), ("Fine-tuned", pred_ft), ("Scratch CNN", pred_scratch)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=class_names).plot(ax=a, cmap="Blues", colorbar=False, xticks_rotation=45)
    a.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# Look at some mistakes of the fine-tuned model
wrong = np.where(pred_ft != y_test)[0][:8]
plt.figure(figsize=(12, 6))
for i, idx in enumerate(wrong):
    img = tf.io.decode_jpeg(tf.io.read_file(p_test[idx]), channels=3)
    plt.subplot(2, 4, i + 1); plt.imshow(tf.image.resize(img, IMG_SIZE).numpy().astype("uint8"))
    plt.title(f"true: {class_names[y_test[idx]]}\npred: {class_names[pred_ft[idx]]}", fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()

## 7. Performance Analysis
*(Write this section in your own words after running the notebook, using your numbers. Points to cover:)*
- **Frozen vs fine-tuned:** how much did accuracy / macro-F1 change after unfreezing the top 30 layers? Did validation loss improve or start to overfit?
- **Transfer learning vs scratch:** compare the pretrained model to the small CNN. Pretrained features (edges, textures, shapes learned from ImageNet) usually give much higher accuracy with the same data and far less training time.
- **Per-class behaviour:** which classes get confused in the confusion matrix (e.g. roses vs tulips) and why (visual similarity, background clutter, label noise)? The misclassified images above help here.
- **Cost:** only the small head is trained in phase 1, so it converges in a few epochs.

## 8. Conclusion
Summarise in 4-5 sentences: the final test accuracy, whether fine-tuning was worth it, challenges you hit (e.g. BatchNorm handling, choosing learning rates, overfitting on a small dataset) and what you learned about transfer learning.

**References:** Sandler et al., *MobileNetV2: Inverted Residuals and Linear Bottlenecks*, CVPR 2018. TensorFlow Flowers dataset (TensorFlow team). Keras Applications documentation.